### Import Modules

In [265]:
import pandas as pd,numpy as np
from sklearn.svm import SVC,LinearSVC,NuSVC
from sklearn.model_selection import train_test_split,cross_val_score, StratifiedKFold,GridSearchCV
from sklearn.preprocessing import LabelEncoder,StandardScaler,MinMaxScaler,normalize
from sklearn.metrics import accuracy_score,classification_report
import joblib
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from umap import UMAP

### Load Dataset

In [90]:
data = pd.read_csv("./embeddings_result/embeddings_call_graph_clusters_v1.csv")
data['code_file'] = data['code_file'].apply(lambda x: x.replace('/root/AI-Pattern-Mining-Project/outputs/prompt & rag/20251028_085918 - Run/generated_code/',''))
data['pattern'] = data['code_file'].apply(lambda x: x.split('/')[0])

In [14]:
data.head()

,code_file,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,...,dim_759,dim_760,dim_761,dim_762,dim_763,dim_764,dim_765,dim_766,dim_767,pattern
0,Advanced MT Prompting/pattern_1.py,0.010473,-0.086456,0.020704,0.072811,-0.069658,0.079100,0.016109,-0.116357,0.035185,...,-0.024537,-0.067877,0.075597,0.099720,0.035907,0.233789,-0.317256,0.070238,0.092933,Advanced MT Prompting
1,Advanced MT Prompting/pattern_10.py,0.002774,-0.155825,-0.016958,0.109435,-0.134882,-0.026956,0.019212,-0.135563,0.029505,...,-0.018589,-0.053212,0.127422,0.065531,0.024898,0.390377,-0.205732,0.016368,0.122040,Advanced MT Prompting
2,Advanced MT Prompting/pattern_11.py,-0.000473,-0.199095,-0.024379,0.095847,-0.138792,-0.012554,0.002148,-0.176035,0.042417,...,-0.024046,-0.045352,0.143503,0.057276,0.029717,0.420159,-0.237789,0.003915,0.154167,Advanced MT Prompting
3,Advanced MT Prompting/pattern_12.py,0.016031,-0.193469,0.001521,0.100525,-0.102650,-0.121538,0.017241,-0.197112,0.069094,...,0.019782,-0.054770,0.114624,0.046622,0.047399,0.395274,-0.168212,-0.044403,0.211058,Advanced MT Prompting
4,Advanced MT Prompting/pattern_13.py,0.003953,-0.221190,-0.005675,0.139310,-0.175301,-0.049305,0.008009,-0.185112,0.046989,...,-0.029435,-0.020810,0.141135,0.036710,0.048247,0.451174,-0.169562,-0.032311,0.211697,Advanced MT Prompting


### Helper Functions

In [40]:
def print_scores(y_test, y_pred,target_names):
    print(classification_report(y_test, y_pred, target_names=target_names))

### Preprocess Data

In [95]:
le = LabelEncoder()
data['pattern_encoded'] = le.fit_transform(data['pattern'])
features_cols = [col for col in data.columns if col not in ['code_file','pattern','pattern_encoded']]

In [308]:
X = data[features_cols]
y = data['pattern_encoded']
# X = normalize(X)
X = StandardScaler().fit_transform(X)


### Feature Selection

In [309]:
from sklearn.feature_selection import SelectKBest, f_classif
selector = SelectKBest(f_classif, k=442)
X = selector.fit_transform(X, y)
X = StandardScaler().fit_transform(X)

### Train Test Split

In [310]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### SVC Exp

In [312]:
svc_model = SVC(kernel='linear', probability=True, random_state=42)
svc_model.fit(X_train, y_train)

svc_pred = svc_model.predict(X_test)
print_scores(y_test, svc_pred, target_names=le.classes_)

                                                                           precision    recall  f1-score   support

                                                    Advanced MT Prompting       1.00      0.75      0.86         4
                                         Bias Mitigation & Output Quality       1.00      0.75      0.86         4
                                              Cross-lingual LLM Prompting       1.00      1.00      1.00         4
                                       Enhanced User Intent Comprehension       0.80      1.00      0.89         4
                                          Explainable AI (XAI) Techniques       1.00      1.00      1.00         4
                                          Integrating External Knlowladge       0.50      0.50      0.50         4
                                             Integrating Knlowladge Graph       0.80      1.00      0.89         4
                                        Iterative Optimizations and ReAct      

In [271]:
params_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'gamma':[1,0.1,0.01,0.001,'scale','auto']
}

grid = GridSearchCV(SVC(),param_grid=params_grid,refit=True,cv=5,verbose=2)
grid.fit(X_train, y_train)
print("Best Parameters:",grid.best_params_)

Fitting 5 folds for each of 72 candidates, totalling 360 fits
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   0.1s
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   0.1s
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   0.1s
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   0.1s
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   0.0s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.1s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.1s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.1s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.1s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.1s
[CV] END ........................C=0.1, gamma=1, kernel=poly; total time=   0.0s
[CV] END ........................C=0.1, gamma=1

In [307]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2,)
svc_cv_scores = cross_val_score(svc_model, X, y, cv=kf, scoring='f1_weighted',)
print(f"SVC Cross-Validation F1 Scores: {svc_cv_scores}")
print(f"SVC Mean CV F1 Score: {np.mean(svc_cv_scores)}")

SVC Cross-Validation F1 Scores: [0.75953151 0.74906065 0.73327109 0.72464562 0.80517848]
SVC Mean CV F1 Score: 0.7543374693289028


### Other Model EXP

In [107]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

In [108]:
def model_eval(model,name=''):
    cv_scores = cross_val_score(model, X, y, cv=kf, scoring='f1_weighted')
    print(f"Summary of model: ")
    print(f" - SVC Cross-Validation F1 Scores: {cv_scores}")
    print(f" - SVC Mean CV F1 Score: {np.mean(cv_scores)}")

In [109]:
model = KNeighborsClassifier(n_neighbors=2)
model_eval(model)

Summary of model: 
 - SVC Cross-Validation F1 Scores: [0.43815629 0.46660177 0.40706302 0.53799534 0.3940124 ]
 - SVC Mean CV F1 Score: 0.44876576415037944


In [110]:
model = RandomForestClassifier(n_estimators=26, random_state=1)
model_eval(model)

Summary of model: 
 - SVC Cross-Validation F1 Scores: [0.47051282 0.54034918 0.49280442 0.5719253  0.56321734]
 - SVC Mean CV F1 Score: 0.5277618108387339


In [111]:
model = DecisionTreeClassifier(random_state=1)
model_eval(model)

Summary of model: 
 - SVC Cross-Validation F1 Scores: [0.2707931  0.24960872 0.23336386 0.3535742  0.35392385]
 - SVC Mean CV F1 Score: 0.2922527472527473


### Final SVC Model Build and Save

In [112]:
final_svc_model = SVC(kernel='linear', C=1)
final_svc_model.fit(X, y)

,C,1
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [81]:
joblib.dump(final_svc_model, './models/svc_classification_model_v1.joblib')
joblib.dump(le, './models/label_encoder_v1.joblib')

['./models/label_encoder_v1.joblib']